# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b16b11a5-5422-4985-ab6c-f35456b6c587;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [3]:
from pyspark.sql import functions as F

# 1. Déclarer les 5 tables Bronze de football
BRONZE_TABLES = {
    "main": "/lakehouse/bronze/transfers",
    "players": "/lakehouse/bronze/players",
    "clubs": "/lakehouse/bronze/clubs",
    "leagues": "/lakehouse/bronze/leagues",
    "club_league": "/lakehouse/bronze/club_league"
}

# 2. Charger les tables Delta Lake depuis HDFS
bronze_dfs = {
    alias: spark.read.format("delta").load(path)
    for alias, path in BRONZE_TABLES.items()
}

# 3. Afficher le nombre de lignes et les colonnes
for alias, df in bronze_dfs.items():
    print(
        f"[{alias}] {BRONZE_TABLES[alias]} -> "
        f"{df.count():,} lignes | colonnes : {df.columns[:8]}"
    )

# 4. Sélectionner la table principale : les transferts
df_bronze = bronze_dfs["main"]

# 5. Calculer le nombre de valeurs NULL par colonne
null_report = df_bronze.select([
    F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

# 6. Afficher le diagnostic
null_report.show(truncate=False)

[main] /lakehouse/bronze/transfers -> 3,469,314 lignes | colonnes : ['player_id', 'year', 'left_club_id', 'joined_club_id', 'transfer_fee', 'transfer_type', '_source_file', '_ingested_at']
[players] /lakehouse/bronze/players -> 858,947 lignes | colonnes : ['id', 'url', 'name', 'full_name', 'birth_date', 'death_date', 'height', 'citizenship_1']
[clubs] /lakehouse/bronze/clubs -> 12,963 lignes | colonnes : ['id', 'url', 'club_name', 'id_current_league', 'current_mv', '_source_file', '_ingested_at', '_source_dataset']
[leagues] /lakehouse/bronze/leagues -> 475 lignes | colonnes : ['id', 'url', 'league_name', 'country', 'current_mv', '_source_file', '_ingested_at', '_source_dataset']
[club_league] /lakehouse/bronze/club_league -> 127,980 lignes | colonnes : ['club_id', 'league_id', 'season', 'squad', 'market_value', '_source_file', '_ingested_at', '_source_dataset']
+---------+----+------------+--------------+------------+-------------+------------+------------+---------------+
|player_id|

## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [6]:
from pyspark.sql import functions as F

# Chemin de notre future table Silver
SILVER_MAIN_PATH = "/lakehouse/silver/football_transfers"

# ÉTAPE 1 : Nettoyer les transferts
df_transfers = bronze_dfs["main"]

df_transfers_clean = (
    df_transfers
    .dropDuplicates()
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("transfer_fee", F.col("transfer_fee").cast("double"))
)

count_before_join = df_transfers_clean.count()

print("Transferts après nettoyage :", count_before_join)


# ÉTAPE 2 : Préparer les tables secondaires
df_players = (
    bronze_dfs["players"]
    .select(
        F.col("id").alias("player_key"),
        F.col("name").alias("player_name")
    )
    .dropDuplicates(["player_key"])
)

df_clubs = (
    bronze_dfs["clubs"]
    .select(
        F.col("id").alias("club_key"),
        F.col("club_name")
    )
    .dropDuplicates(["club_key"])
)


# ÉTAPE 3 : Effectuer les jointures
df_silver = (
    df_transfers_clean

    # Relier chaque transfert à son joueur
    .join(
        df_players,
        F.col("player_id") == F.col("player_key"),
        "left"
    )
    .drop("player_key")

    # Relier chaque transfert au club rejoint
    .join(
        F.broadcast(df_clubs),
        F.col("joined_club_id") == F.col("club_key"),
        "left"
    )
    .drop("club_key")

    # Ajouter deux indicateurs métier
    .withColumn(
        "is_paid_transfer",
        F.col("transfer_fee") > 0
    )
    .withColumn(
        "club_identified",
        F.col("club_name").isNotNull()
    )
)

# Vérifier le plan d'exécution Spark
df_silver.explain("simple")


# ÉTAPE 4 : Enregistrer au format Delta Lake
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

# Vérifier le résultat
silver_count = (
    spark.read
    .format("delta")
    .load(SILVER_MAIN_PATH)
    .count()
)

assert silver_count <= count_before_join, (
    f"Alerte explosion N-M : {silver_count:,} lignes "
    f"après jointure > {count_before_join:,} avant jointure !"
)

print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes")

Transferts après nettoyage : 3469314
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [player_id#2118, year#2119, left_club_id#2120, joined_club_id#2121, transfer_fee#2840, transfer_type#2123, _source_file#2124, _ingested_at#2125, _source_dataset#2126, player_name#2938, club_name#2940, (transfer_fee#2840 > 0.0) AS is_paid_transfer#2935, isnotnull(club_name#2940) AS club_identified#2936]
   +- BroadcastHashJoin [joined_club_id#2121], [club_key#2934], LeftOuter, BuildRight, false, false
      :- Project [player_id#2118, year#2119, left_club_id#2120, joined_club_id#2121, transfer_fee#2840, transfer_type#2123, _source_file#2124, _ingested_at#2125, _source_dataset#2126, player_name#2938]
      :  +- BroadcastHashJoin [player_id#2118], [player_key#2932], LeftOuter, BuildRight, false, false
      :     :- HashAggregate(keys=[joined_club_id#2121, _source_dataset#2126, transfer_type#2123, player_id#2118, left_club_id#2120, year#2119, _ingested_at#2125, transfer_fee#2122, _sour

Table Silver écrite dans /lakehouse/silver/football_transfers : 3,469,314 lignes


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [7]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794913640913,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"946cbd4f-a4ef-48d7-a032-c4c8ab9c86d9","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-bca29876-a7d9-4ed2-8fdd-6452d7ed7b18-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-3b937ca5-083f-4673-9472-6e9f0f9d0d28-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-1e91b501-4cc2-4f11-a7ed-d353c3f1a5b7-c000.snappy.parquet","partition

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [8]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# 1. Ouvrir la table Silver
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# 2. Lire les données actuelles
df_current = spark.read.format("delta").load(SILVER_MAIN_PATH)

# Conserver la version actuelle avant modification
version_before = target_delta.history(1).select("version").first()[0]

# 3. Sélectionner un transfert dont le type est Unknown
# Utiliser les colonnes d'origine pour identifier la ligne
keys = [
    "player_id", "year", "left_club_id",
    "joined_club_id", "transfer_fee", "transfer_type",
    "_source_file", "_ingested_at", "_source_dataset"
]

correction = (
    df_current
    .filter(F.col("transfer_type") == "Unknown")
    .select(*keys)
    .limit(1)
    .withColumn("new_transfer_type", F.lit("Unknown - test MERGE"))
)

assert correction.count() == 1, "Aucun transfert Unknown trouvé."

# 4. Construire la condition de correspondance
condition = " AND ".join(
    f"t.`{c}` <=> s.`{c}`" for c in keys
)

# 5. Appliquer le MERGE INTO
(
    target_delta.alias("t")
    .merge(correction.alias("s"), condition)
    .whenMatchedUpdate(set={
        "transfer_type": "s.new_transfer_type"
    })
    .execute()
)

print("MERGE INTO exécuté avec succès !")

# 6. Afficher l'historique des modifications
history_df = spark.sql(
    f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`"
)

history_df.select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=80)

# 7. Lire la version avant modification
df_old = (
    spark.read.format("delta")
    .option("versionAsOf", version_before)
    .load(SILVER_MAIN_PATH)
)

# 8. Lire la version actuelle
df_new = spark.read.format("delta").load(SILVER_MAIN_PATH)

print("Avant MERGE :", df_old.filter(
    F.col("transfer_type") == "Unknown - test MERGE"
).count())

print("Après MERGE :", df_new.filter(
    F.col("transfer_type") == "Unknown - test MERGE"
).count())

print("Lignes avant :", df_old.count())
print("Lignes après :", df_new.count())

26/11/17 14:20:25 WARN MapPartitionsRDD: RDD 346 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


MERGE INTO exécuté avec succès !
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1| 2026-11-17 14:20:23.94|    MERGE|{numTargetRowsCopied -> 277814, numTargetRowsDeleted -> 0, numTargetBytesRemo...|
|      0|2026-11-17 11:08:17.173|    WRITE|{numFiles -> 13, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows ->...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+



Avant MERGE : 0


Après MERGE : 1
Lignes avant : 3469314
Lignes après : 3469314
